# 05 · Evaluation: one trained run judged on clients and a period it never saw

**What it does.** Scores a trained run on two **parts** it never saw: the **test split** (the
clients held out of training, over the observation window) and the **validation period** (after
the window). It sets the model beside the declared baselines, explains what the model learned,
and writes the model results report and the model card. Nothing is fitted or chosen here:
Evaluation only measures, outside the model.

**How one evaluation goes:**
1. The config is resolved and the run is found: the one you name, or by default the latest
   trained run of this data definition with the current Training settings.
2. The bundle is loaded and checked; the test and validation rows are scored inside Spark and
   written as the scores table.
3. The scores are summed into score cells; every metric, slice and stratum is computed from
   them, each with a client-bootstrap interval.
4. The baselines are scored and set beside the model.
5. The explanation sample (seeded test rows) gets its contributions, permutation importance and
   partial dependence.
6. The drift reference for Monitoring, the report and the model card are written, then a `run`
   entry in the decision audit log.

**Before you start.** Run `04_training`: Evaluation needs a trained run. It reads only the run
folder and the data, so a later EDA or Feature Checks rerun never changes what an older run's
evaluation shows.

**Automation levels:** Evaluation runs the same at both. At both, a person signs off the model's
features after reading this notebook and before the model is registered (step 12).

**What it writes**, in the run folder's `evaluation/`: `evaluation.json`, `tables/` (the
metrics, slices, curves, strata, baselines, the explanation tables, the score cells and the drift
reference), `scores/` (every scored row), `report.html`, `model_card.json` and
`model_card.html`.

**No gates.** Nothing here says passed or failed: the numbers, their intervals and the baselines
are the evidence a person judges.

**Setup** (the next cell): the same names as `04_training`; for the other example, change
`CONFIG`. It prints the decision and the evaluation settings: the headline metric, the baselines
and the slices, all fixed before any result is seen.

**Docs:** `docs/components/evaluation.md` · `docs/components/reporting.md` ("The model results
report", "The model card") · previous: `04_training`.

In [ ]:
import json
from pathlib import Path

import shap
from IPython.display import HTML, Markdown, display

import core_modelling
from core_modelling.config import load_config
from core_modelling.environment import get_spark_session, load_environment
from core_modelling.evaluation import compare_runs, run_evaluation
from core_modelling.runs import list_runs, show_run

# The use case and where the framework runs, as absolute paths. The repo's examples are found
# from the installed package; a use-case repo writes its own config path here and may read the
# environment file from $CORE_MODELLING_ENVIRONMENT instead.
REPO = Path(core_modelling.__file__).resolve().parents[2]
CONFIG = REPO / "conf/base/cc_acquisition.yml"  # or conf/base/td_acquisition.yml
ENTITY = "entity_a"
ENVIRONMENT = str(REPO / "conf/local/environment.yml")  # or None: $CORE_MODELLING_ENVIRONMENT

env = load_environment(ENVIRONMENT)
spark = get_spark_session(env)
cfg = load_config(CONFIG)
print(cfg.use_case, cfg.task, cfg.automation_level)
print(cfg.decision)
print(cfg.evaluation)

## 1. Pick a run
**`list_runs`**: the use case's runs on this entity, newest first, one row each, with Training's
summary, Evaluation's once it ran, and the states `trained`, `evaluated` and `signed_off`.

**`RUN = None`** evaluates the default: the latest **trained** run of this data definition with
the current Training settings and sources. To evaluate an older run, name it from the `run`
column (`"r001"`; `r1` and `1` work too). It must be trained and of the same data definition,
client split and sources; a refusal says why. A run with `trained` false is incomplete and
cannot be evaluated.

Evaluation applies today's `evaluation` settings (the slices, the intervals, the views left
out), so a run can be evaluated again after you change them, without training again.

In [ ]:
display(list_runs(env, cfg, ENTITY))
RUN = None  # the latest trained run of this data definition, or a run number such as "r001"

## 2. Evaluate
`run_evaluation` finds the run, scores it and writes `evaluation/`; `result.tables` holds
Evaluation's tables, `result.meta` `evaluation.json`'s content and `result.views` the report's
seventeen views. It prints the frames it collected, any warning, then `evaluation of r…
written` and the folder.

The cell then makes each view a list of blocks to display (a subsection's title becomes a
heading), prints the run's record (`show_run`: Evaluation's summary, `evaluated` now true) and
where `report.html` is: one self-contained page with every view and an appendix (the resolved
config, the audit log, the frames collected, the seconds per step). It opens without internet;
share it as it is.

**When it fails**, the run is left as it was: nothing is written before the end. The message
names the fix: no trained run of this data definition (run `04_training`), a named run of
another data definition, client split or sources, or a name in `evaluation.skip_views` that is
no view.

**A rerun** rewrites `evaluation/` whole and adds one `run` entry to the audit log; the bundle,
`training/` and the per-run ledger are never touched.

In [ ]:
result = run_evaluation(cfg, ENTITY, env, spark, run=RUN)
views = {
    name: [
        HTML(item) if isinstance(item, str) else item
        for block in blocks
        for item in (
            (Markdown(f"#### {block[0]}"), *block[1]) if isinstance(block, tuple) else (block,)
        )
    ]
    for name, blocks in result.views.items()
}
print(show_run(env, cfg, ENTITY, result.meta["run_id"]))
print(result.paths["report.html"])

## 3. The summary
Read it first: it says what was evaluated and what was chosen before you read any number.
- **The opening line**: the run, its model (`model_hash`), the parts scored, and the unsampled
  train split's rate that the skill scores are measured against.
- **The headline**: the decision's metric (`decision.metric`; `log_loss` in this example) with
  AUC beside it (regression: the concordance), on test and on validation, each with its 95%
  interval.
- **Choices**: the decision, the baselines, the slices, the interval settings, the validation
  period; all declared before any result.
- **Baselines**: the model beside each baseline at the decision's metric and AUC, on both parts.
  `population_base_rate` ranks nothing (AUC 0.5): it anchors the skill scores.
  `single_feature` is the feature with the highest train concordance, used as the score: the
  model's gain over it is what the other features add.
- **How to read**: concordance, lift, the intervals, grey and null.
- **What these numbers cannot say**: Training's out-of-fold metric is optimistic; the gap
  between train and test; EDA and Feature Checks saw test labels, so validation is the one
  honest number; validation is reported, never chosen on; the validation periods earlier
  evaluations scored; a base rate that moved inside the window.
- **Warnings**: only those that apply.

**Good:** the model above both baselines on test, its interval clear of the single feature's;
test and validation intervals that overlap. **Worrying:**
- validation far below test: the period differs from the window, or a feature behaves
  differently after it;
- train (in sample) far above test: the model fits its own clients much better than new ones;
- the model barely above the single feature: the other features add little;
- an AUC that looks too good: look for a feature that is the target under another name (step 8).

In [ ]:
display(*views["summary"])

## 4. Metrics per period
Every metric per part, pooled and per observation period, the validation period beside them and
the mean of the periods in the last row; a period below `strata.min_events` events is greyed,
never hidden. The chart draws the headline metric per period with its interval.

**The metrics** (yes/no target):
- **AUC**: how well the score orders clients, 0.5 chance, 1 perfect.
- **Average precision** (`ap`): read beside the base rate, its floor; not comparable across use
  cases.
- **Log-loss and Brier skill**: 1 minus the loss over the loss of always predicting the train
  rate: 0 no better than that rate, 1 perfect, negative worse.
- **Observed to expected**: the events over the sum of the scores; 1 is the right level.
- **Calibration slope and intercept**: 1 and 0 when the scores are calibrated.

Regression shows RMSE, MAE, R², the deviance, the concordance and predicted to observed.

**Look at:** whether a metric is stable across the periods; a steady slide towards the
validation period means the relation is changing over time. Observed to expected moving away
from 1 period by period means the base rate moved; Evaluation does not correct for it.

In [ ]:
display(*views["metrics"])

## 5. Curves, capture and the slices
- **ROC and precision-recall**, one chart each per part, on all of the part's rows (regression:
  not available, the target has no classes).
- **Capture**: the share of all events (regression: of the total amount) held by the top x% of
  scores, per part.
- **The slice table**: the declared slices (`evaluation.slices`: here the top 1%, the top 10%
  and the top 10,000), then the depths 1 to 50%. Each is cut inside each period, then the
  periods are added up: rows, events captured, capture, the slice's rate (precision) and its
  lift over the part's rate, with intervals. A `top_n` is a share of the period's whole eligible
  population, so it reads as a percentage on each part.

**Look at:** the slice your decision acts on: its capture and lift on test, then validation. A
lift at the top 1% far above the lift at 10% means the signal sits at the very top of the list.

In [ ]:
display(*views["curves"], *views["capture"])

## 6. Reliability and the score distribution
- **Reliability**: the score against the observed rate in 10 groups of the score, per part; on
  the diagonal is calibrated. The table beside it: observed to expected, the calibration slope
  and intercept, pooled per part. Under 200 events a note says the groups are noisy: read the
  intervals. Regression: predicted against observed in 20 groups.
- **Score distribution**: the score's distribution for positives and for the rest, each to its
  own scale, per part. The further apart, the better the ranking.

**Look at:** for a `probability` decision, calibration is the point: test points near the
diagonal and observed to expected near 1. Validation off while test is on means the rate moved
in the validation period, not that the calibration is wrong. Test off as well: try another
`training.calibration.method` in `04_training` step 9 (it is fitted on train only), then
evaluate the new run and compare (step 11).

In [ ]:
display(*views["reliability"], *views["score_distribution"])

## 7. Strata
Per variable of `strata.variables` (here `segment` and `onboarding_channel`) and per level, on
each part: rows, events, base rate, the AUC inside the level, observed to expected, and at the
decision's slice the level's share of the list beside its share of the population, with the
capture inside it. A level below `strata.min_events` events is greyed. There are no tests across
levels: read many levels together and some will differ by chance.

**Worrying:**
- an AUC inside a level near 0.5 while the pooled AUC is high: the model sorts the levels, not
  the clients within them;
- observed to expected far from 1 in a level: the model's level is off there;
- a level taking far more of the list than of the population: say so to whoever uses the list.

In [ ]:
display(*views["strata"])

## 8. The explanations
What the model learned, from the **explanation sample**: seeded test rows (the same at every
rerun) plus three marked rows for the waterfalls. A **contribution** is how far a feature moved
one row's margin from the base value; a row's contributions and the base value add up to its
margin. They are the model library's own, equal to SHAP's tree explainer, on the margin's scale
(the log-odds for a yes/no target, the log scale for the `tweedie` and `poisson` objectives),
before calibration.

**Importance and the beeswarm:**
- **The importance bars**: each feature's share of the mean absolute contribution on test,
  beside its train share from Training's out-of-fold sample. Close shares mean the model leans on
  the same features for new clients. One feature far ahead of all others: is it known at scoring
  time, and not the target under another name?
- **The beeswarm**: one line per feature, one point per sampled row at its contribution,
  coloured by the feature's value (blue low, red high, grey null; a categorical in one colour,
  hover for its level). Red on the right means higher values raise the score.

The members of one redundancy cluster share their importance: read them together.

In [ ]:
display(*views["importance"], *views["beeswarm"])

### The effects, beside EDA's lift
For each **effect feature** (the model's ten largest by out-of-fold contribution share, plus the
EDA key features the model uses, twenty at most): left, the mean contribution of the sample's
rows in each of EDA's groups of the feature, with its interval (greyed with too few rows);
right, EDA's lift by the same groups on its snapshot. A categorical adds a table by level. A
feature without EDA's groups shows its contribution against its value instead.

**Good:** the two rise and fall together. **Worrying:** the model's effect against EDA's
direction. The feature may act through others (a correlated feature takes the plain effect), or
the relation may not be what it seems; either way, it needs a reason at the sign-off (step 12).

In [ ]:
display(*views["effects"])

### The waterfalls
Three rows of test: the highest score, a row at the decision's cut (the top 10% when no slice is
declared) and a typical row. Each walks from the base value to the row's margin, one bar per
feature (the nine largest, the rest in one bar), labelled `feature = value`; red raises the
score, blue lowers it.

**Use them** to explain one score in words to a business reader. **Check** that the highest
score is not driven by one implausible value.

In [ ]:
display(*views["waterfalls"])

### Permutation importance and partial dependence
- **Permutation importance per cluster**: on the explanation sample, the members of each
  redundancy cluster are shuffled together and the loss change is measured, so near-duplicates do
  not split the credit. Beside it, the cluster's contribution share and both ranks; the line
  above gives Spearman's correlation of the two orders, and `disagrees` marks the three largest
  rank gaps. A high correlation means the two methods agree. A large contribution with a small
  loss change: another feature carries the same information.
- **Partial dependence and ICE** for the effect features: each feature set in turn to every point
  of its grid on every sample row, the rest kept; the thick line is the mean score, each thin
  line one row. Lines that cross mean the feature acts together with others.

In [ ]:
display(*views["permutation_importance"], *views["pdp_ice"])

### `shap`'s own plots, if you want them
The report draws its explanation charts itself, in `shap`'s colours. For `shap`'s own plots,
build a `shap.Explanation` from the stored contributions table: its `<feature>__contribution`
columns, the base value `base_value` and the features' raw values. The numbers are the same as
in the views above. Any of `shap.plots` then works: `bar(explanation)`,
`waterfall(explanation[0])`, `scatter(explanation[:, "<feature>"])`. This is for exploring
only: nothing a run or a report reads.

In [ ]:
sample = result.tables["contributions"]
suffix = "__contribution"
features = [name.removesuffix(suffix) for name in sample if name.endswith(suffix)]
explanation = shap.Explanation(
    sample[[f"{name}{suffix}" for name in features]].to_numpy(),
    base_values=sample["base_value"].to_numpy(),
    data=sample[features].to_numpy(),
    feature_names=features,
)
shap.plots.beeswarm(explanation)

## 9. The rest of the report
- **Search**: Training's search per family and its progress chart, as in `04_training` step 3,
  with Training's seconds per step.
- **Thresholds**: per part, the score threshold where the list first reaches each depth of the
  capture view, with the rows at or above it, the capture, the rate, the lift and the false
  positive rate: what a decision that cuts on the score would take.
- **Validation on test clients only**: the validation period's metrics on test-split clients
  alone, unseen clients in an unseen period. Close to the full validation numbers means the train
  clients in the validation period do not flatter it.

`evaluation.skip_views` leaves any view but the summary out of the report and of `views`; delete
its name here then.

In [ ]:
display(*views["search"], *views["thresholds"], *views["validation_test_clients"])

## 10. The model card
`model_card.json`, and the same facts as a small page in `model_card.html`, beside the report.
It is built from the run folder alone: **generated facts only**, no free text, never typed by
hand, rewritten with each evaluation. Its sections:
- **details**: the run, the hashes, the framework and library versions, the family, the dates;
- **intended use**: the decision, the slices, the population, the target and its window;
- **data**: the sources, the window, the validation period, rows and events per period, the
  sampling, the split;
- **features**: the final list (kind, source, share, direction), the held features with their
  flags, the excluded with their reasons, the kept with their findings;
- **training**: the families searched, the trials, the recipe, the calibration;
- **performance**: the metric rows and the baselines comparison, with no gates;
- **considerations**: the warnings raised, as codes with their values;
- **sign-off** and **lifecycle**: empty until the model is registered.

In [ ]:
card = json.loads(Path(result.paths["model_card.json"]).read_text())
print(result.paths["model_card.html"])
display(list(card), card["details"])

## 11. Compare two runs
After a change (a `keep`, an `exclude`, a budget: `04_training` step 9) you train again,
evaluate the new run here, then ask "did it help?". `compare_runs(cfg, ENTITY, env, spark,
run_a, run_b)` reads both runs' stored scores. Both must be evaluated runs of this config's data
definition; runs of two data definitions are refused. Both are measured on the rows both scored,
each client with the same bootstrap weights in both, so the interval of a difference is right. A
difference is b minus a.
- **`rows`**: per part, each run's rows, the shared rows and how many scored differently.
- **`metrics`**: test, the two runs side by side, each value with its interval.
- **`differences`**: **test only**, per metric: which way is better, the difference, its
  interval and the verdict (`better`, `worse`, `no clear difference` when the interval holds
  zero, or `not available` with the reason).
- **`validation`**: both runs' validation numbers, also on test clients only, side by side under
  "reported, never chosen on", with no difference. Choose on test: a choice read off validation
  turns it into a second test and leaves no honest number.
- **`features`**: the features that entered or left; **`importance`**: the largest moves in
  importance rank.
- **`budgets`**: each run's trials and seconds per family beside its headline on the shared test
  rows: did more trials buy anything?

As shipped, the cell compares the run with itself: every difference is exactly zero, a check
that the comparison runs. Set `RUN_A` to an earlier run, evaluated first (step 1 with its number,
then step 2).

In [ ]:
RUN_A, RUN_B = result.meta["run_id"], result.meta["run_id"]  # a: the earlier run, b: the new one
compared = compare_runs(cfg, ENTITY, env, spark, RUN_A, RUN_B)
display(compared["rows"], compared["differences"], compared["metrics"], compared["validation"])
display(compared["features"], compared["importance"], compared["budgets"])

## 12. The sign-off
At every automation level, a person signs off every feature of the model before it is
registered; an LLM never does. It is the one gate between a trained run and registering it. The
view below is Training's sign-off view (train-only evidence, as in `04_training` step 7); read
it with the test evidence above: the importance, the effects beside EDA, the waterfalls.

**Check, feature by feature:** known at scoring time; not the target under another name; a
direction that makes sense (a disagreement with EDA needs a reason); a similar share on train and
test.
- **Every feature defensible:** record the sign-off. It records the run's final feature list and
  its signature, and covers every run of the same data definition with the same final features,
  earlier ones included; `list_runs`' `signed_off` and `show_run`'s states then say so.
- **A feature fails:** an `exclude` with stage `sign_off` and the reason; then train again
  (`04_training`), evaluate the new run, compare (step 11) and sign off the new run.

Paste a call into your own cell (`CONFIG`, `ENTITY` and `ENVIRONMENT` are the setup cell's):
```python
from core_modelling.config.audit_log import record_decision
from core_modelling.runs import sign_off

sign_off(cfg, ENTITY, env, "<run>", reason="<why every feature is defensible>")
record_decision(CONFIG, ENTITY, exclude=["<feature>"], stage="sign_off",
                reason="<why it must not be used>", env_path=ENVIRONMENT)
```
Or the CLI: `python -m core_modelling decide conf/base/cc_acquisition.yml --entity entity_a
--sign-off --run <run> --reason "<why>"`.

**What comes next:** registering a signed-off run as an immutable, hashed version, then
promoting it, comes with `06_register_promote`, not built yet.

In [ ]:
display(*views["sign_off"])

## Before you take the run further
- The run evaluated is the one you meant: its number and `model_hash` in the summary.
- The model is above the single-feature baseline on test, its interval clear of the baseline's.
- Test and validation agree within their intervals, or the gap is explained (a rate that moved
  shows in observed to expected per period).
- Every warning in the summary is understood.
- For a `probability` decision, observed to expected is near 1 on test.
- No stratum level where the model orders no better than chance without a reason.
- No feature dominates the importance that could be the target under another name; train and
  test shares are close.
- The effects agree with EDA's lift, or each disagreement has a reason.
- Any choice between runs was made on test; validation was read, never chosen on.
- A person has signed off every feature: `signed_off` is true in `list_runs`.
- The report and the model card are read and shared.